# 실습 3: MNIST 파이프라인 Apply (응용 및 실험)

이번 실습에서는 데이터셋 EDA부터 전처리, 피처 엔지니어링, 그리고 증강(Augmentation) 기법들을 직접 비교하고 실험해 봅니다.
`lab_02`에서 배운 원리를 바탕으로 코드를 한 단계씩 분해하며 각 실험의 의도를 깊이 파악합니다.

[!Open In Colab](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/3주차/lab_03_apply.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

# 재현성을 위한 시드 고정 및 디바이스 설정
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

### 🔬 코드 해설
- `lab_01`, `lab_02`와 동일하게 딥러닝 텐서 연산을 위한 `torch`, 시각화를 위한 `matplotlib.pyplot` 등을 불러옵니다.
- 이 실습에서는 여러 번 모델을 학습시키는 파이프라인이 반복되므로, 학습 진행 상황 모니터링을 위해 `tqdm`을 активно 활용합니다.

## 2. MNIST 데이터셋 EDA (탐색적 데이터 분석)

In [ ]:
# 원본 데이터 로드 (Transform 없이)
raw_train_dataset = datasets.MNIST(root='./data', train=True, download=True)
raw_test_dataset = datasets.MNIST(root='./data', train=False, download=True)

# 1-1. 데이터 크기 및 이미지 특성 확인
print(f"Train dataset size: {len(raw_train_dataset)}")
print(f"Test dataset size: {len(raw_test_dataset)}")

img, label = raw_train_dataset[0]
print(f"Image mode: {img.mode}, Image size: {img.size}")

# 학습 데이터 평균, 표준편차 계산 (연산을 위해 임시로 텐서 변환)
tensor_transform = transforms.ToTensor()
train_tensors = torch.stack([tensor_transform(img) for img, _ in raw_train_dataset])
mean = train_tensors.mean()
std = train_tensors.std()
print(f"\nTrain Data Mean: {mean.item():.4f}")
print(f"Train Data Std: {std.item():.4f}")

# 1-2. 학습 데이터의 숫자별 분포 시각화
train_labels = [label for _, label in raw_train_dataset]
label_counts = Counter(train_labels)

plt.figure(figsize=(8, 4))
plt.bar(label_counts.keys(), label_counts.values(), color='skyblue', edgecolor='black')
plt.title("Distribution of Classes in Train Dataset")
plt.xlabel("Class (Digit)")
plt.ylabel("Frequency")
plt.xticks(range(10))
plt.show()

# 1-3. 추가 EDA: 클래스별 첫 번째 이미지 시각화
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.flatten()
for i in range(10):
    idx = train_labels.index(i)
    img, label = raw_train_dataset[idx]
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Class: {i}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

### 🔬 코드 해설
- 학습 파이프라인 설계 전 가장 먼저 수행해야 하는 데이터 탐색(EDA) 과정입니다.
- 데이터의 평균(`0.1307`)과 표준편차(`0.3081`)를 계산합니다. `lab_01`과 `lab_02`에서 사용한 정규화 수치가 하늘에서 떨어진 매직 넘버가 아니라 실제 데이터 통계량임을 증명합니다.
- 클래스 불균형이 있는지 시각적으로 점검합니다. MNIST는 숫자 0~9가 고르게 분포되어 있어 별도의 밸런싱 작업이 필요 없음을 알 수 있습니다.

## 3. 비교 실험을 위한 공통 파이프라인 함수

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

def train_and_evaluate(train_transform, test_transform, epochs=3):
    # transform이 적용된 데이터셋 로드
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

    train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

    model = SimpleMLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    
    # 학습 루프
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        # 여러 번 실험이 실행되므로 leave=False 로 설정하여 출력창을 깔끔하게 유지합니다.
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for data, target in pbar:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)

    # 평가 루프
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
    accuracy = 100 * correct / total
    return train_losses, accuracy

### 🔬 코드 해설
- 앞으로 여러 전처리와 증강을 반복 테스트해야 하므로, 중복 코드를 방지하고자 데이터 로딩, 모델 생성, 훈련, 평가를 `train_and_evaluate` 함수 하나로 묶었습니다.
- `train_transform`과 `test_transform`을 별도의 파라미터로 받습니다. (학습 데이터와 평가 데이터의 변환 규칙은 서로 달라야 하기 때문입니다.)
- `tqdm(..., leave=False)` 옵션을 사용해 에폭별 막대가 완료되면 사라지게 만들어 여러 번의 실험 출력 결과가 화면을 어지럽히지 않도록 배려했습니다.

## 4. 전처리 유무에 따른 학습 성능 비교
모델의 성능은 데이터의 질에 크게 좌우됩니다. 데이터를 모델이 소화하기 좋은 형태로 다듬는 전처리 과정의 효과를 단계별로 실험합니다.

### 4-1. 전처리 없는 코드 (정규화 X)
이미지를 그저 0~1 사이의 값으로만 변환(`ToTensor()`)하고, 별도의 평균/표준편차 정규화는 수행하지 않은 베이스라인입니다.

In [ ]:
print("--- 4-1. 정규화 X ---")
transform_no_prep = transforms.ToTensor()
loss_no_prep, acc_no_prep = train_and_evaluate(transform_no_prep, transform_no_prep)
print(f"정규화 X - Test Accuracy: {acc_no_prep:.2f}%")

**🔬 코드 해설:** 값의 범위가 0~1에 몰려 있어, 활성화 함수와 파라미터 업데이트가 한쪽 방향으로 치우치기 쉽습니다. 학습은 되지만 최적의 경로는 아닐 수 있습니다.

### 4-2. 전처리 있는 코드 (데이터셋 통계 기반 정규화 O)
앞서 구한 MNIST 학습 데이터의 실제 평균(0.1307)과 표준편차(0.3081)를 사용하여 정규화합니다.

In [ ]:
print("\n--- 4-2. 정규화 O (0.1307, 0.3081) ---")
transform_with_prep = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
loss_with_prep, acc_with_prep = train_and_evaluate(transform_with_prep, transform_with_prep)
print(f"정규화 O (0.1307, 0.3081) - Test Accuracy: {acc_with_prep:.2f}%")

**🔬 코드 해설:** 데이터의 분포가 평균 0, 분산 1을 중심으로 재편됩니다. 모든 입력 특징의 스케일이 비슷해져 옵티마이저가 Loss의 최솟값을 훨씬 빠르고 안정적으로 찾아갈 수 있습니다.

### 4-3. 또 다른 전처리 (일괄 0.5 정규화)
데이터셋의 통계량을 모를 때 임시방편으로 많이 쓰는 방식으로, 값의 범위를 대략 [-1, 1]로 맞춥니다.

In [ ]:
print("\n--- 4-3. 정규화 O (0.5, 0.5) ---")
transform_other_prep = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
loss_other_prep, acc_other_prep = train_and_evaluate(transform_other_prep, transform_other_prep)
print(f"정규화 O (0.5, 0.5) - Test Accuracy: {acc_other_prep:.2f}%")

**🔬 코드 해설:** 정확한 데이터셋의 통계를 쓴 것(`4-2`)만큼은 아닐지라도, 정규화가 없는 것(`4-1`)보다는 훨씬 딥러닝 모델의 초기 파라미터(보통 평균 0 주변에 초기화됨)와 잘 맞아떨어집니다.

### 4-4. 전처리 방식에 따른 Loss 감소 시각화
세 가지 전처리 방식의 학습 속도(Loss 감소)를 시각적으로 비교해 봅니다.

In [ ]:
plt.plot(range(1, 4), loss_no_prep, marker='o', label='No Norm')
plt.plot(range(1, 4), loss_with_prep, marker='o', label='Norm (0.13, 0.30)')
plt.plot(range(1, 4), loss_other_prep, marker='o', label='Norm (0.5, 0.5)')
plt.title('Training Loss Comparison (Preprocessing)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks([1, 2, 3])
plt.legend()
plt.grid(True)
plt.show()

**🔬 시각화 해설:** 통계량을 활용한 정규화(`Norm (0.13, 0.30)`)가 동일한 에폭 대비 가장 Loss를 빠르고 낮게 떨어뜨리는 것을 확인할 수 있습니다. 좋은 전처리는 모델 구조를 바꾸는 것 이상의 효과를 냅니다.

## 5. 피처 엔지니어링 결과 비교
데이터에 대한 사람의 '도메인 지식'을 넣어 모델이 집중해야 할 특징을 돋보이게 만드는 과정을 실험합니다.

### 5-1. 베이스라인 성능 확인
비교를 위해 가장 좋았던 전처리(`4-2`)의 성능을 기준점으로 삼습니다.

In [ ]:
print(f"원본 특징 (베이스라인) - Test Accuracy: {acc_with_prep:.2f}%")

### 5-2. 피처 엔지니어링 1 (이진화 - Binarization)
숫자의 형태, 즉 '윤곽선'만 중요할 것이라는 가설 하에 흐릿한 픽셀 값을 날려버리고 극단적인 흑과 백(0과 1)으로 치환해 봅니다.

In [ ]:
print("\n--- 5-2. 피처 엔지니어링 (이진화) ---")
transform_binarize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x > 0.5).float()), # 0.5 기준으로 0 또는 1
    transforms.Normalize((0.1307,), (0.3081,))
])
_, acc_binarize = train_and_evaluate(transform_binarize, transform_with_prep)
print(f"피처 엔지니어링 (이진화) - Test Accuracy: {acc_binarize:.2f}%")

**🔬 코드 해설:** `transforms.Lambda`를 사용하면 PyTorch가 제공하지 않는 나만의 변환 규칙을 마음대로 적용할 수 있습니다. 하지만 때로는 부드러운 필압(명암) 정보가 유용했을 수도 있기에 성능이 오를지 내릴지는 실험적 확인이 필요합니다.

### 5-3. 피처 엔지니어링 2 (색상 반전 - Invert)
글씨가 검은색이고 배경이 흰색으로 반전된다면 모델은 동일하게 잘 학습할까요?

In [ ]:
print("\n--- 5-3. 피처 엔지니어링 (색상 반전) ---")
transform_invert = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 1.0 - x), # 1에서 빼서 색상 반전
    transforms.Normalize((0.1307,), (0.3081,))
])
_, acc_invert = train_and_evaluate(transform_invert, transform_with_prep)
print(f"피처 엔지니어링 (색상 반전) - Test Accuracy: {acc_invert:.2f}%")

**🔬 코드 해설:** 신경망은 값의 '패턴'을 학습하므로 색상이 반전되어도 충분히 학습이 가능합니다. 그러나 원본 평가 데이터(`transform_with_prep`)와 전혀 다른 양상의 입력을 받게 되므로 이질적인 특징으로 인해 성능 저하가 발생할 수 있음을 관찰합니다.

## 6. 데이터 증강(Augmentation) 학습 결과 비교
한정된 데이터를 이리저리 비틀어 모델에게 '이런 변형도 여전히 같은 숫자란다'라고 알려주는 데이터 증강의 강도와 효과를 실험합니다.

### 6-1. 평가용(Test) 파이프라인 및 베이스라인
데이터 증강을 수행할 때 **가장 중요한 원칙은 평가 데이터(Test Data)에는 증강을 적용하지 않는 것**입니다. 실전에서 들어올 순수 이미지를 평가해야 하기 때문입니다.

In [ ]:
# 평가 데이터는 공정성을 위해 증강을 적용하지 않습니다.
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

print(f"증강 X (베이스라인) - Test Accuracy: {acc_with_prep:.2f}%")

### 6-2. 약한 회전 (5도 회전)
사람이 숫자를 쓸 때 삐뚤게 쓰는 정도인 5도를 랜덤하게 적용하여, 모델이 약간의 기울어짐에도 강건(Robust)해지도록 유도합니다.

In [ ]:
print("\n--- 6-2. 약한 회전 (5도) ---")
train_transform_rot_5 = transforms.Compose([
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
_, acc_rot_5 = train_and_evaluate(train_transform_rot_5, test_transform)
print(f"약한 회전 (5도) - Test Accuracy: {acc_rot_5:.2f}%")

**🔬 코드 해설:** `RandomRotation`은 이미지를 불러올 때마다 지정된 각도 내에서 랜덤하게 이미지를 회전시킵니다. 적절한 증강은 일반화(Generalization) 성능을 높여 Test Accuracy 향상에 기여할 수 있습니다.

### 6-3. 강한 회전 (45도 회전)
만약 회전 각도를 45도까지 강하게 주면 어떻게 될까요?

In [ ]:
print("\n--- 6-3. 강한 회전 (45도) ---")
train_transform_rot_45 = transforms.Compose([
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
_, acc_rot_45 = train_and_evaluate(train_transform_rot_45, test_transform)
print(f"강한 회전 (45도) - Test Accuracy: {acc_rot_45:.2f}%")

**🔬 코드 해설:** 45도나 돌아간 6은 9로 보일 수도 있고, 숫자 고유의 특성을 잃어버릴 수 있습니다. **데이터의 도메인을 무시한 과도한 증강은 오히려 노이즈**가 되어 학습을 크게 방해한다는 것을 직접 숫자로 확인했습니다.

### 6-4. 미세한 이동과 크기 변환 (Random Affine)
회전 외에도, 카메라 렌즈에서 사물이 약간 이동하거나 멀어지는 현상을 모사하는 Affine 변환을 적용합니다.

In [ ]:
print("\n--- 6-4. 이동 및 크기 변환 (Random Affine) ---")
train_transform_affine = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
_, acc_affine = train_and_evaluate(train_transform_affine, test_transform)
print(f"이동/크기 변환 (Random Affine) - Test Accuracy: {acc_affine:.2f}%")

**🔬 코드 해설:** `translate=(0.1, 0.1)`은 가로/세로로 10% 범위 내에서 랜덤 이동을, `scale=(0.9, 1.1)`은 90~110% 크기 확대를 수행합니다. 글씨를 중앙에서 벗어나게 쓰거나 크기가 제각각인 상황에 모델이 유연하게 대처하도록 돕는 유용한 컴퓨터 비전 증강 기법 중 하나입니다.

## 7. ✅ 실습 결과 정리
- `lab_03`에서는 데이터를 본격적으로 주무르며 학습 결과가 어떻게 달라지는지 확인했습니다.
- EDA를 통해 데이터의 특성과 통계량을 파악하고, 모델의 입력 기준(`Normalize`)을 세웠습니다.
- `tqdm`을 적용한 파이프라인 함수 위에서 피처 엔지니어링과 데이터 증강을 실험했습니다.
- 🎯 **핵심 결론:** 적절한 전처리와 증강은 모델 성능 향상의 핵심이지만, 데이터 도메인을 무시한 과도한 변환은 성능 저하를 초래합니다. 또한 `train`과 `test`의 transform 분리가 왜 필수적인지 명확히 이해해야 합니다.